In [23]:
import pandas as pd
import numpy as np

In [24]:
df = pd.read_csv("arXiv_scientific_dataset.csv")

In [25]:
df.head()

,id,title,category,category_code,published_date,updated_date,authors,first_author,summary,summary_word_count
0,cs-9308101v1,Dynamic Backtracking,Artificial Intelligence,cs.AI,8/1/93,8/1/93,['M. L. Ginsberg'],'M. L. Ginsberg',Because of their occasional need to return to ...,79
1,cs-9308102v1,A Market-Oriented Programming Environment and ...,Artificial Intelligence,cs.AI,8/1/93,8/1/93,['M. P. Wellman'],'M. P. Wellman',Market price systems constitute a well-underst...,119
2,cs-9309101v1,An Empirical Analysis of Search in GSAT,Artificial Intelligence,cs.AI,9/1/93,9/1/93,"['I. P. Gent', 'T. Walsh']",'I. P. Gent',We describe an extensive study of search in GS...,167
3,cs-9311101v1,The Difficulties of Learning Logic Programs wi...,Artificial Intelligence,cs.AI,11/1/93,11/1/93,"['F. Bergadano', 'D. Gunetti', 'U. Trinchero']",'F. Bergadano',As real logic programmers normally use cut (!)...,174
4,cs-9311102v1,Software Agents: Completing Patterns and Const...,Artificial Intelligence,cs.AI,11/1/93,11/1/93,"['J. C. Schlimmer', 'L. A. Hermens']",'J. C. Schlimmer',To support the goal of allowing users to recor...,187


In [26]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 287421 entries, 0 to 287420
Data columns (total 10 columns):
 #   Column              Non-Null Count   Dtype
---  ------              --------------   -----
 0   id                  287421 non-null  str  
 1   title               287421 non-null  str  
 2   category            287421 non-null  str  
 3   category_code       287421 non-null  str  
 4   published_date      287421 non-null  str  
 5   updated_date        287421 non-null  str  
 6   authors             287421 non-null  str  
 7   first_author        287421 non-null  str  
 8   summary             287421 non-null  str  
 9   summary_word_count  287421 non-null  int64
dtypes: int64(1), str(9)
memory usage: 21.9 MB


In [27]:
df.shape

(287421, 10)

In [28]:
df.isnull().sum()

id                    0
title                 0
category              0
category_code         0
published_date        0
updated_date          0
authors               0
first_author          0
summary               0
summary_word_count    0
dtype: int64

In [29]:

df.duplicated().sum()

np.int64(0)

In [30]:
df = df.drop_duplicates()

In [31]:
df.duplicated().sum()

np.int64(0)

In [32]:
df.columns

Index(['id', 'title', 'category', 'category_code', 'published_date',
       'updated_date', 'authors', 'first_author', 'summary',
       'summary_word_count'],
      dtype='str')

In [33]:
df["summary"] = df["summary"].str.lower()

In [34]:
import re

df["summary"] = df["summary"].apply(lambda x: re.sub(r"[^\w\s]", "", x))

In [35]:
df["summary"] = df["summary"].apply(lambda x: re.sub(r"\d+", "", x))

In [36]:
df[["title","summary"]].head()

,title,summary
0,Dynamic Backtracking,because of their occasional need to return to ...
1,A Market-Oriented Programming Environment and ...,market price systems constitute a wellundersto...
2,An Empirical Analysis of Search in GSAT,we describe an extensive study of search in gs...
3,The Difficulties of Learning Logic Programs wi...,as real logic programmers normally use cut an...
4,Software Agents: Completing Patterns and Const...,to support the goal of allowing users to recor...


In [37]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Sara\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [38]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

df["summary"] = df["summary"].apply(
    lambda x: " ".join(word for word in x.split() if word not in stop_words)
)

In [39]:
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Sara\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Sara\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [40]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

df["summary"] = df["summary"].apply(
    lambda x: " ".join(lemmatizer.lemmatize(word) for word in x.split())
)

In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df["summary"])

print("Shape of TF-IDF Matrix:", X.shape)

Shape of TF-IDF Matrix: (287421, 5000)


In [42]:
import sklearn
print(sklearn.__version__)

1.9.0


In [43]:
from sklearn.metrics.pairwise import cosine_similarity

# Example user query
query = "machine learning in healthcare"

# Convert the query into TF-IDF vector
query_vector = tfidf.transform([query])

# Calculate similarity between query and all research papers
similarity_scores = cosine_similarity(query_vector, X).flatten()

# Get indices of top 5 most similar papers
top_indices = similarity_scores.argsort()[-5:][::-1]

# Display the results
results = df.iloc[top_indices][["title", "summary", "category"]].copy()
results["similarity_score"] = similarity_scores[top_indices]

results

,title,summary,category,similarity_score
19874,A Survey on Federated Learning for the Healthc...,recent technological advancement considerately...,Computers and Society,0.610128
35310,Machine Learning Applications In Healthcare: T...,detection easily missed hidden pattern fast pr...,Machine Learning,0.535786
94927,Beyond Volume: The Impact of Complex Healthcar...,medical chart national census healthcare tradi...,Computers and Society,0.522931
112789,Creating Trustworthy LLMs: Dealing with Halluc...,large language model proliferated across multi...,Computation and Language (Natural Language Pro...,0.515227
17025,Healthsheet: Development of a Transparency Art...,machine learning ml approach demonstrated prom...,Artificial Intelligence,0.507050


In [44]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df["summary"])

print("TF-IDF ready:", X.shape)

TF-IDF ready: (287421, 5000)


In [45]:
query = "deep learning for medical diagnosis"

query_vector = tfidf.transform([query])

similarity_scores = cosine_similarity(query_vector, X).flatten()

top_indices = similarity_scores.argsort()[-5:][::-1]

results = df.iloc[top_indices][["title", "category"]].copy()
results["similarity_score"] = similarity_scores[top_indices]

results

,title,category,similarity_score
175115,Sim4Seg: Boosting Multimodal Multi-disease Med...,Computer Vision and Pattern Recognition,0.605707
22709,Medical Dialogue Generation via Intuitive-then...,Computation and Language (Natural Language Pro...,0.571274
170531,Diagnosing Shoulder Disorders Using Multimodal...,Computer Vision and Pattern Recognition,0.563878
19812,How can Deep Learning Retrieve the Write-Missi...,Computation and Language (Natural Language Pro...,0.558696
57234,Web Applicable Computer-aided Diagnosis of Gla...,Computer Vision and Pattern Recognition,0.542243


In [46]:
def search_papers(query, top_n=10):
    query_vector = tfidf.transform([query])

    similarity_scores = cosine_similarity(
        query_vector, X
    ).flatten()

    top_indices = similarity_scores.argsort()[-top_n:][::-1]

    results = df.iloc[top_indices][
        ["title", "category", "authors", "summary"]
    ].copy()

    results["similarity_score"] = similarity_scores[top_indices]

    return results

In [47]:
search_papers("deep learning for medical diagnosis", 5)

,title,category,authors,summary,similarity_score
175115,Sim4Seg: Boosting Multimodal Multi-disease Med...,Computer Vision and Pattern Recognition,"['Lingran Song', 'Yucheng Zhou', 'Jianbing Shen']",despite significant progress pixellevel medica...,0.605707
22709,Medical Dialogue Generation via Intuitive-then...,Computation and Language (Natural Language Pro...,"['Kaishuai Xu', 'Wenjun Hou', 'Yi Cheng', 'Jia...",medical dialogue system attracted growing rese...,0.571274
170531,Diagnosing Shoulder Disorders Using Multimodal...,Computer Vision and Pattern Recognition,"['Jindong Hong', 'Wencheng Zhang', 'Shiqin Qia...",shoulder disorder frozen shoulder aka adhesive...,0.563878
19812,How can Deep Learning Retrieve the Write-Missi...,Computation and Language (Natural Language Pro...,"['Shaohui Liu', 'Xien Liu', 'Ji Wu']",purpose writemissing diagnosis detection find ...,0.558696
57234,Web Applicable Computer-aided Diagnosis of Gla...,Computer Vision and Pattern Recognition,"['Mijung Kim', 'Olivier Janssens', 'Ho-min Par...",glaucoma major eye disease leading vision loss...,0.542243


In [48]:
def search_papers(query, top_n=10, min_score=0.0):
    query_vector = tfidf.transform([query])

    similarity_scores = cosine_similarity(
        query_vector, X
    ).flatten()

    top_indices = similarity_scores.argsort()[-top_n:][::-1]

    results = df.iloc[top_indices][
        ["title", "category", "authors", "summary"]
    ].copy()

    results["similarity_score"] = similarity_scores[top_indices]

    results = results[
        results["similarity_score"] >= min_score
    ]

    return results

In [49]:
search_papers("artificial intelligence in healthcare", 10, 0.3)

,title,category,authors,summary,similarity_score
8158,Comprehensible Artificial Intelligence on Know...,Artificial Intelligence,"['Simon Schramm', 'Christoph Wehner', 'Ute Sch...",artificial intelligence application gradually ...,0.542515
9045,Can transformative AI shape a new age for our ...,Artificial Intelligence,"['Jesus L. Lobo', 'Javier Del Ser']",artificial intelligence widely regarded transf...,0.542321
19874,A Survey on Federated Learning for the Healthc...,Computers and Society,"['Ali Kashif Bashir', 'Nancy Victor', 'Sweta B...",recent technological advancement considerately...,0.535871
7147,The Governance of Physical Artificial Intellig...,Artificial Intelligence,"['Yingbo Li', 'Anamaria-Beatrice Spulber', 'Yu...",physical artificial intelligence prove one imp...,0.525285
3153,Three IQs of AI Systems and their Testing Methods,Artificial Intelligence,"['Feng Liu', 'Yong Shi', 'Ying Liu']",rapid development artificial intelligence brou...,0.509930
4319,Stovepiping and Malicious Software: A Critical...,Artificial Intelligence,"['Jason M. Pittman', 'Jesus P. Espinoza', 'Cou...",awareness possible impact associated artificia...,0.466430
23678,Foundation Model for Advancing Healthcare: Cha...,Computers and Society,"['Yuting He', 'Fuxiang Huang', 'Xinrui Jiang',...",foundation model pretrained broad data able ad...,0.450241
172310,An Approach to Joint Hybrid Decision Making be...,Human-Computer Interaction,"['Jonas D. Rockbach', 'Sven Fuchs', 'Maren Ben...",due progress artificial intelligence important...,0.450149
6711,Metaverse for Healthcare: A Survey on Potentia...,Artificial Intelligence,"['Rajeswari Chengoden', 'Nancy Victor', 'Thien...",rapid progress digitalization automation led a...,0.448525
17420,Securing AI-based Healthcare Systems using Blo...,Cryptography and Security,"['Rucha Shinde', 'Shruti Patil', 'Ketan Kotech...",healthcare system increasingly incorporating a...,0.441959


In [51]:
import joblib
from scipy.sparse import save_npz

joblib.dump(tfidf, "tfidf_vectorizer.pkl")
save_npz("tfidf_matrix.npz", X)

print("Model saved successfully!")

Model saved successfully!


In [ ]:
--